# Sales Forecasting and Demand Prediction

This notebook is intentionally output-free and requires a real CSV. It mirrors the repository pipeline without fabricating data.


In [ ]:
import os
from pathlib import Path

import pandas as pd

SEED = 42
TARGETS = ['Units Sold', 'Demand Forecast']
DATA_PATH = Path(os.environ.get('SALES_DATA_PATH', 'data/retail_store_inventory.csv')).expanduser()
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}. Set SALES_DATA_PATH; no data was generated.')
df = pd.read_csv(DATA_PATH)


In [ ]:
required = {'Date', *TARGETS}
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError(f'Dataset is missing required columns: {missing}')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
if df['Date'].isna().any():
    raise ValueError('Dataset contains invalid or missing Date values.')
for target in TARGETS:
    df[target] = pd.to_numeric(df[target], errors='coerce')
    if df[target].isna().any():
        raise ValueError(f'Target column {target!r} contains missing/non-numeric values.')
df = df.sort_values('Date', kind='stable').reset_index(drop=True)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

split = int(len(df) * 0.8)
train, test = df.iloc[:split], df.iloc[split:]
def features(frame):
    x = frame.drop(columns=TARGETS).copy()
    date = pd.to_datetime(x.pop('Date'))
    x['date_year'] = date.dt.year
    x['date_month'] = date.dt.month
    x['date_dayofweek'] = date.dt.dayofweek
    x['date_day'] = date.dt.day
    return x.drop(columns=['Inventory_Demand', 'UnitsSold_Price'], errors='ignore')
X_train, X_test = features(train), features(test)
y_train, y_test = train[TARGETS], test[TARGETS]
numeric = X_train.select_dtypes(include=['number']).columns.tolist()
categorical = [c for c in X_train.columns if c not in numeric]
preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])


In [ ]:
# Optional training/evaluation cell; it runs only after the real-data checks above.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = Pipeline([('preprocess', preprocess), ('regressor', ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
metrics = {}
for i, target in enumerate(TARGETS):
    metrics[target] = {'mae': mean_absolute_error(y_test[target], pred[:, i]), 'rmse': mean_squared_error(y_test[target], pred[:, i]) ** 0.5, 'r2': r2_score(y_test[target], pred[:, i])}
metrics
